In [ ]:
import matplotlib.pyplot as plt
import sys, os
import numpy as np
from scipy.sparse.linalg import spsolve
from scipy.sparse import csr_matrix
import time

sys.path.append(os.path.abspath("../src"))
import initial_conditions
import constants
import newton
import state
import residual
import jacobian
import newton

In [ ]:
r_first = constants.FLOATDTYPE(0.001)

constants.R_OUTER = constants.FLOATDTYPE(200.0)
constants.N_SHELL = 200

In [ ]:
ln_r_edge, u = initial_conditions.set_up_initial_conditions(r_first)

In [ ]:
constants.SIGMA_OVER_M = constants.FLOATDTYPE(0.3)
constants.BETA = constants.FLOATDTYPE(0.8)
constants.ALPHA = constants.FLOATDTYPE(1.0)

In [ ]:
state_init = state.state_from_unknowns(ln_r_edge, u)

In [ ]:
#covergence
constants.F_TOL = constants.FLOATDTYPE(1.0e-12)

#stagnated but acceptable
constants.F_ACCEPT = constants.FLOATDTYPE(1.0e-8)

#determines stagnation
constants.X_TOL = constants.FLOATDTYPE(1.0e-12)

#maximum allowed iterations
constants.ITER_MAX = 50

#timestep controller
constants.DT_TOL = constants.FLOATDTYPE(0.01)

#total time limiter
constants.D_STOP = constants.FLOATDTYPE(1.0e4)

In [ ]:
dt_trial = 0.001

t = 0
state_curr = state_init

start_time = time.perf_counter()

while np.exp(state_curr["ln_rho"]).max() <= constants.D_STOP:

    dt_step = dt_trial

    u_old = state_curr["u"]
    r_edge_old = state_curr["r_edge"]

    state_new = newton.iterate(state_curr, dt_step)

    if state_new is None:
        raise RuntimeError("Newton failed")

    u_new = state_new["u"]
    r_edge_new = state_new["r_edge"]

    eps_r = np.max(np.abs(r_edge_new[1:-1] - r_edge_old[1:-1]) / r_edge_old[1:-1])
    eps_u = np.max(np.abs(u_new - u_old) / u_old)

    eps = max(eps_r, eps_u)

    if eps > constants.DT_TOL:

        # reject this timestep
        dt_trial = dt_step * constants.FLOATDTYPE(0.9) * constants.DT_TOL / eps

        print("reject:", "dt =", dt_step, "eps =", eps, "new dt =", dt_trial)

        continue

    # Accept the timestep.
    state_curr = state_new
    t += dt_step

    dt_trial = dt_step * constants.FLOATDTYPE(0.9) * constants.DT_TOL / eps

    print(
        "accept:",
        "t=",
        t,
        "dt =",
        dt_step,
        "eps =",
        eps,
        "new dt =",
        dt_trial,
        "rho_0=",
        np.exp(state_curr["ln_rho"]).max(),
    )

end_time = time.perf_counter()

# 4. Calculate total execution time
execution_time = end_time - start_time

In [ ]:
execution_time